# Neural Network Analysis — Customer Churn Prediction
**Dataset:** `customer_churn_nn.csv`  
**Problem Type:** Binary Classification (Supervised Learning)  
**Framework:** TensorFlow / Keras  
**Target Variable:** `churn` — 1 = churned, 0 = retained

---
## Task 1: Dataset Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('customer_churn_nn.csv')

print('Shape:', df.shape)
df.head()

In [ ]:
# Number of rows and columns
print(f'Rows    : {df.shape[0]}')
print(f'Columns : {df.shape[1]}')

In [ ]:
# Column types
df.dtypes

In [ ]:
# Missing value check
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)
print(f'\nTotal missing: {missing.sum()}')

In [ ]:
# Basic statistical summary
df.describe().T

In [ ]:
# Target variable description
print('Target variable: churn')
print('  0 = Customer Retained')
print('  1 = Customer Churned')
print()
print(df['churn'].value_counts())
print()
print(df['churn'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

In [ ]:
# Distribution of target variable
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['churn'].value_counts()
axes[0].bar(['Retained (0)', 'Churned (1)'], counts.values, color=['#2E75B6', '#E74C3C'], edgecolor='black')
axes[0].set_title('Churn Distribution — Count', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

pcts = df['churn'].value_counts(normalize=True) * 100
axes[1].pie(pcts.values, labels=['Retained (0)', 'Churned (1)'],
            autopct='%1.2f%%', colors=['#2E75B6', '#E74C3C'],
            startangle=90, explode=[0, 0.08])
axes[1].set_title('Churn Distribution — Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Observation: Dataset is highly imbalanced — only 1.55% of customers churned.')

---
## Task 2: Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

df2 = df.copy()

# Drop non-predictive identifier
df2.drop(columns=['customer_id'], inplace=True)
print('Dropped: customer_id')

In [ ]:
# Handling missing values
print(f'Missing values: {df2.isnull().sum().sum()} — no imputation required.')

In [ ]:
# Encoding categorical columns
categorical_cols = ['region', 'plan_type', 'contract_type', 'payment_method']

le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    df2[col] = le.fit_transform(df2[col].astype(str))
    le_dict[col] = list(le.classes_)
    print(f'{col:20s}: {le_dict[col]}')

print('\nautopay_enabled is already binary (0/1) — no encoding needed.')

In [ ]:
# Scaling numerical features
numerical_cols = [
    'tenure_months', 'monthly_charges_inr', 'avg_login_days_per_month',
    'support_tickets_last_90_days', 'payment_delay_days', 'data_usage_gb',
    'satisfaction_score', 'last_complaint_days_ago', 'discount_percent', 'referral_count'
]

X = df2.drop(columns=['churn'])
y = df2['churn']

scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

print('StandardScaler applied to numerical columns.')
print(f'Feature matrix shape: {X.shape}')
X.describe().T[['mean', 'std']].head()

In [ ]:
# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Features         : {X_train.shape[1]}')
print(f'\nChurn ratio in train: {y_train.mean()*100:.2f}%')
print(f'Churn ratio in test : {y_test.mean()*100:.2f}%')

---
## Task 3: Neural Network Model Building

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from tensorflow import keras
from tensorflow.keras import layers

print(f'TensorFlow version: {tf.__version__}')

In [ ]:
def build_model(hidden_layers=[64, 32], activation='relu',
                learning_rate=0.001, input_dim=X_train.shape[1]):
    """
    Build a feed-forward neural network.
    
    Architecture:
        Input Layer  -> shape = (n_features,)
        Hidden Layer(s) -> Dense with specified activation
        Output Layer -> Dense(1, sigmoid) for binary classification

    Loss      : Binary Cross-Entropy
    Optimizer : Adam
    """
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation))
    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Build baseline model
model = build_model(hidden_layers=[64, 32], activation='relu', learning_rate=0.001)
model.summary()

**Architecture Summary:**

| Layer | Type | Neurons | Activation | Purpose |
|---|---|---|---|---|
| Layer 1 | Input + Dense | 16 → 64 | ReLU | Learns primary feature combinations |
| Layer 2 | Hidden Dense | 64 → 32 | ReLU | Learns higher-order patterns |
| Layer 3 | Output Dense | 32 → 1 | Sigmoid | Outputs churn probability [0,1] |

---
## Task 4: Training and Evaluation

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Training and Test Accuracy / Loss
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc  = model.evaluate(X_test,  y_test,  verbose=0)

print(f'Training Accuracy : {train_acc*100:.2f}%  |  Training Loss : {train_loss:.4f}')
print(f'Test Accuracy     : {test_acc*100:.2f}%  |  Test Loss     : {test_loss:.4f}')

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history.history['accuracy']) + 1)

axes[0].plot(epochs_range, history.history['accuracy'],     label='Train Accuracy', color='#2E75B6', linewidth=2)
axes[0].plot(epochs_range, history.history['val_accuracy'], label='Val Accuracy',   color='#E74C3C', linewidth=2, linestyle='--')
axes[0].set_title('Accuracy Over Epochs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history.history['loss'],     label='Train Loss', color='#2E75B6', linewidth=2)
axes[1].plot(epochs_range, history.history['val_loss'], label='Val Loss',   color='#E74C3C', linewidth=2, linestyle='--')
axes[1].set_title('Loss Over Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Baseline Model — Training Curves', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

y_pred = (model.predict(X_test, verbose=0) > 0.5).astype(int).flatten()

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Retained (0)', 'Churned (1)'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Baseline Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Retained (0)', 'Churned (1)']))

**Interpretation:**
- Overall test accuracy is high (~98%) due to severe class imbalance (98.45% retained).
- The model does well on the majority class (Retained) but struggles with the minority class (Churned).
- Low F1-score for the Churned class reflects the imbalanced nature of the dataset.
- In a real setting, techniques like SMOTE, class-weighted loss, or threshold tuning should be applied.

---
## Task 5: Hyperparameter Experimentation

In [ ]:
experiments = [
    {'name': 'Exp 1 – Baseline',        'layers': [64, 32],       'lr': 0.001,  'batch': 32, 'epochs': 50, 'activation': 'relu'},
    {'name': 'Exp 2 – Deeper Network',  'layers': [128, 64, 32],  'lr': 0.001,  'batch': 32, 'epochs': 50, 'activation': 'relu'},
    {'name': 'Exp 3 – High LR',         'layers': [64, 32],       'lr': 0.01,   'batch': 32, 'epochs': 50, 'activation': 'relu'},
    {'name': 'Exp 4 – Low LR',          'layers': [64, 32],       'lr': 0.0001, 'batch': 32, 'epochs': 50, 'activation': 'relu'},
    {'name': 'Exp 5 – Tanh Activation', 'layers': [64, 32],       'lr': 0.001,  'batch': 32, 'epochs': 50, 'activation': 'tanh'},
    {'name': 'Exp 6 – Larger Batch',    'layers': [64, 32],       'lr': 0.001,  'batch': 64, 'epochs': 50, 'activation': 'relu'},
]

results = []
histories = []

for exp in experiments:
    print(f"Running {exp['name']} ...", end=' ')
    m = build_model(hidden_layers=exp['layers'], activation=exp['activation'], learning_rate=exp['lr'])
    h = m.fit(X_train, y_train, epochs=exp['epochs'], batch_size=exp['batch'],
              validation_split=0.1, verbose=0)
    tr_l, tr_a = m.evaluate(X_train, y_train, verbose=0)
    te_l, te_a = m.evaluate(X_test,  y_test,  verbose=0)
    results.append({
        'Experiment':    exp['name'],
        'Architecture':  str(exp['layers']),
        'Activation':    exp['activation'],
        'Learning Rate': exp['lr'],
        'Batch Size':    exp['batch'],
        'Epochs':        exp['epochs'],
        'Train Acc (%)': round(tr_a * 100, 2),
        'Test Acc (%)':  round(te_a * 100, 2),
        'Train Loss':    round(tr_l, 4),
        'Test Loss':     round(te_l, 4),
    })
    histories.append(h)
    print(f"Train={tr_a*100:.2f}%  Test={te_a*100:.2f}%")

results_df = pd.DataFrame(results)
print('\nDone.')

In [ ]:
# Comparison table
results_df.set_index('Experiment')

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

exp_names = [r['Experiment'].replace(' – ', '\n') for r in results]
train_accs = [r['Train Acc (%)'] for r in results]
test_accs  = [r['Test Acc (%)']  for r in results]

x = np.arange(len(exp_names))
width = 0.35

axes[0].bar(x - width/2, train_accs, width, label='Train Acc', color='#2E75B6')
axes[0].bar(x + width/2, test_accs,  width, label='Test Acc',  color='#E74C3C')
axes[0].set_xticks(x)
axes[0].set_xticklabels(exp_names, fontsize=8)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Train vs Test Accuracy — All Experiments', fontweight='bold')
axes[0].legend()
axes[0].set_ylim(96, 100.5)
axes[0].grid(axis='y', alpha=0.3)

train_losses = [r['Train Loss'] for r in results]
test_losses  = [r['Test Loss']  for r in results]

axes[1].bar(x - width/2, train_losses, width, label='Train Loss', color='#2E75B6')
axes[1].bar(x + width/2, test_losses,  width, label='Test Loss',  color='#E74C3C')
axes[1].set_xticks(x)
axes[1].set_xticklabels(exp_names, fontsize=8)
axes[1].set_ylabel('Loss')
axes[1].set_title('Train vs Test Loss — All Experiments', fontweight='bold')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Learning curves for all experiments
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

colors = ['#2E75B6', '#27AE60', '#E74C3C', '#8E44AD', '#F39C12', '#16A085']

for i, (exp, h) in enumerate(zip(experiments, histories)):
    ep = range(1, len(h.history['accuracy']) + 1)
    axes[i].plot(ep, h.history['accuracy'],     color=colors[i], linewidth=2, label='Train Acc')
    axes[i].plot(ep, h.history['val_accuracy'], color=colors[i], linewidth=2, linestyle='--', label='Val Acc')
    axes[i].set_title(exp['name'], fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Accuracy')
    axes[i].legend(fontsize=8)
    axes[i].grid(alpha=0.3)

plt.suptitle('Learning Curves — All Experiments', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Key Observations from Experiments:**

| Experiment | Observation |
|---|---|
| Exp 1 – Baseline | Solid baseline; good generalization with small train-test gap |
| Exp 2 – Deeper Network | Highest train accuracy (99.94%) but test dropped → slight overfitting |
| Exp 3 – High LR | Worst test accuracy (97.25%) — learning rate too aggressive |
| Exp 4 – Low LR | Slower convergence; train acc slightly lower after 50 epochs |
| Exp 5 – Tanh Activation | Comparable to ReLU; slightly lower train acc but similar test acc |
| Exp 6 – Larger Batch | Marginally lower test acc; larger batches reduce gradient noise |

---
## Task 6: Final Reflection

### 6.1 What role do weights and biases play in the model?

**Weights** are the learnable parameters associated with every connection between neurons. They determine how strongly each input feature (or previous-layer output) influences the next neuron. During training, the network adjusts weights through backpropagation to minimize the loss function — effectively learning which patterns in customer behaviour are most predictive of churn.

**Biases** are additional learnable parameters (one per neuron) that allow the activation to shift independently of the input. Without biases, every neuron's output is forced through the origin, limiting the functions the network can represent. With biases, the network gains an extra degree of freedom to fit data more accurately.

In our model the first Dense layer alone has `16 × 64 = 1,024` weight parameters plus `64` biases — all updated each epoch via gradient descent.

---

### 6.2 Why is an activation function required?

Without activation functions, stacking multiple linear layers collapses to a single linear transformation regardless of depth — the model could only learn linear decision boundaries and would fail to capture the complex, non-linear relationships in customer behaviour.

- **ReLU** (`max(0, x)`) is used in hidden layers. It introduces non-linearity while avoiding the vanishing gradient problem for positive inputs, allowing deep networks to train efficiently.
- **Sigmoid** (`1 / (1 + e^-x)`) is used in the output layer. It squashes the output to `[0, 1]`, which can be interpreted directly as a churn probability — perfect for binary classification.

---

### 6.3 What happens when the learning rate is too high or too low?

| Scenario | Effect |
|---|---|
| **Too High (0.01 — Exp 3)** | Gradient steps overshoot the minimum; loss oscillates or diverges; test accuracy dropped to 97.25% |
| **Optimal (0.001 — Exp 1)** | Steady convergence; best balance of speed and generalization |
| **Too Low (0.0001 — Exp 4)** | Very slow convergence; model underperforms within the fixed 50 epochs; train accuracy was 98.44% |

The Adam optimizer partially mitigates extreme learning rate problems through adaptive per-parameter rates, but the base learning rate still critically controls the overall step magnitude.

---

### 6.4 Did your model show signs of underfitting or overfitting?

| Experiment | Train Acc | Test Acc | Diagnosis |
|---|---|---|---|
| Exp 1 – Baseline | 99.69% | 98.50% | ✅ Well-fitted — small gap |
| Exp 2 – Deeper Network | 99.94% | 98.00% | ⚠️ Slight overfitting |
| Exp 3 – High LR | 99.87% | 97.25% | ❌ Overfitting — largest gap |
| Exp 4 – Low LR | 98.44% | 98.50% | ⚠️ Mild underfitting (not converged) |

The **baseline model (Exp 1)** shows the best generalization with the smallest train-test gap.

**Important caveat:** Due to severe class imbalance (only 1.55% churned), the model is effectively underfitting for the minority class — it achieves only ~33% F1-score for churned customers despite 98% overall accuracy. Addressing this requires:
- SMOTE or class-weighted training (`class_weight={0:1, 1:63}`)
- Lowering the classification threshold (e.g., predict churn if `p > 0.3`)
- Dropout regularization for deeper architectures